# Solution A — Training

Trains all Category A candidates (Logistic Regression, SVM, ablations, bootstrap ensemble), compares on the dev set, selects the best model, and saves it to disk.

In [1]:
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    'numpy': 'numpy',
    'pandas': 'pandas',
    'scipy': 'scipy',
    'sklearn': 'scikit-learn',
    'joblib': 'joblib',
}

missing_packages = [
    pip_name
    for module_name, pip_name in REQUIRED_PACKAGES.items()
    if importlib.util.find_spec(module_name) is None
]

if missing_packages:
    print('Installing missing packages:', missing_packages)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing_packages])
else:
    print('All required packages are already available.')


All required packages are already available.


## 1. Imports and path resolution

The notebook should work whether it is run from the repository root or from inside `solution-a/`.

The path helper below searches upward until it finds the coursework root that contains both:

- `training_data/NLI/train.csv`
- `nlu_bundle-feature-unified-local-scorer/`

This avoids the broken relative-path problem that the earlier notebook had.


In [2]:
from __future__ import annotations

import re
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

SEED = 42


def find_project_root() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (
            (candidate / 'training_data' / 'NLI' / 'train.csv').exists()
            and (candidate / 'nlu_bundle-feature-unified-local-scorer').exists()
        ):
            return candidate
    raise FileNotFoundError(
        'Could not find the coursework root. Expected to find training_data/NLI/train.csv '
        'and nlu_bundle-feature-unified-local-scorer/ in the same project tree.'
    )


PROJECT_ROOT = find_project_root()
NOTEBOOK_DIR = PROJECT_ROOT / 'solution-a'
TRAIN_PATH = PROJECT_ROOT / 'training_data' / 'NLI' / 'train.csv'
DEV_PATH = PROJECT_ROOT / 'training_data' / 'NLI' / 'dev.csv'
TRIAL_PATH = PROJECT_ROOT / 'trial_data' / 'NLI_trial.csv'
LOCAL_SCORER_ROOT = PROJECT_ROOT / 'nlu_bundle-feature-unified-local-scorer'
OFFICIAL_BASELINE_PATH = LOCAL_SCORER_ROOT / 'baseline' / '25_DEV_NLI.csv'
ARTEFACT_DIR = NOTEBOOK_DIR / 'artifacts_solution_a'
ARTEFACT_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('PROJECT_ROOT =', PROJECT_ROOT)
print('TRAIN_PATH   =', TRAIN_PATH)
print('DEV_PATH     =', DEV_PATH)
print('TRIAL_PATH   =', TRIAL_PATH)
print('ARTEFACT_DIR =', ARTEFACT_DIR)


PROJECT_ROOT = /Users/jiho/Documents/YR3/34812NLU/NLU_CW
TRAIN_PATH   = /Users/jiho/Documents/YR3/34812NLU/NLU_CW/training_data/NLI/train.csv
DEV_PATH     = /Users/jiho/Documents/YR3/34812NLU/NLU_CW/training_data/NLI/dev.csv
TRIAL_PATH   = /Users/jiho/Documents/YR3/34812NLU/NLU_CW/trial_data/NLI_trial.csv
ARTEFACT_DIR = /Users/jiho/Documents/YR3/34812NLU/NLU_CW/solution-a/artifacts_solution_a


## 2. Data loading

The helper below accepts both training / dev CSVs and trial / test CSVs.

Rules used here:

- `premise` and `hypothesis` are required
- `label` is required only when `require_label=True`
- the helper strips a UTF-8 BOM from the first column name, which is useful for the supplied trial file


In [3]:
def read_pair_dataframe(csv_path: Path, require_label: bool) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    df.columns = [str(col).lstrip('﻿').strip() for col in df.columns]

    required_columns = {'premise', 'hypothesis'}
    missing_required = required_columns - set(df.columns)
    if missing_required:
        raise ValueError(f'{csv_path} is missing required columns: {sorted(missing_required)}')

    if require_label and 'label' not in df.columns:
        raise ValueError(f'{csv_path} must contain a label column for this section of the notebook.')

    if 'label' in df.columns:
        df['label'] = df['label'].astype(int)

    return df


train_df = read_pair_dataframe(TRAIN_PATH, require_label=True)
dev_df = read_pair_dataframe(DEV_PATH, require_label=True)
trial_df = read_pair_dataframe(TRIAL_PATH, require_label=True) if TRIAL_PATH.exists() else None

print('Train rows:', len(train_df))
print('Dev rows:  ', len(dev_df))
print('Trial rows:', len(trial_df) if trial_df is not None else 'trial file not found')
print()
print('Train label distribution:')
print(train_df['label'].value_counts().sort_index())
print()
train_df.head(3)


Train rows: 24432
Dev rows:   6736
Trial rows: 50

Train label distribution:
label
0    11784
1    12648
Name: count, dtype: int64



,premise,hypothesis,label
0,yeah i don't know cut California in half or so...,Yeah. I'm not sure how to make that fit. Maybe...,1
1,actual names will not be used,"For the sake of privacy, actual names are not ...",1
2,The film was directed by Randall Wallace.,The film was directed by Randall Wallace and s...,1


## 3. Evaluation helpers and official baseline table

The local scorer ships with the following metric names for NLI dev evaluation:

- `accuracy_score`
- `macro_precision`
- `macro_recall`
- `macro_f1`
- `weighted_macro_precision`
- `weighted_macro_recall`
- `weighted_mmacro_f1`
- `matthews_corrcoef`

In this notebook, **model selection is driven by `macro_f1`** rather than plain accuracy.
`macro_f1` gives equal weight to both labels, so it is a better fit than plain accuracy when we want balanced performance.

For Category A comparison, we also load the bundled `SVM` predictions from `25_DEV_NLI.csv`.


In [4]:
METRIC_ORDER = [
    'accuracy_score',
    'macro_precision',
    'macro_recall',
    'macro_f1',
    'weighted_macro_precision',
    'weighted_macro_recall',
    'weighted_mmacro_f1',
    'matthews_corrcoef',
]


def metric_summary(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    return {
        'accuracy_score': accuracy_score(y_true, y_pred),
        'macro_precision': precision_score(y_true, y_pred, average='macro', zero_division=0),
        'macro_recall': recall_score(y_true, y_pred, average='macro', zero_division=0),
        'macro_f1': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'weighted_macro_precision': precision_score(y_true, y_pred, average='weighted', zero_division=0),
        'weighted_macro_recall': recall_score(y_true, y_pred, average='weighted', zero_division=0),
        'weighted_mmacro_f1': f1_score(y_true, y_pred, average='weighted', zero_division=0),
        'matthews_corrcoef': matthews_corrcoef(y_true, y_pred),
    }


def metrics_table(predictions: dict[str, np.ndarray], y_true: np.ndarray) -> pd.DataFrame:
    rows = []
    for model_name, y_pred in predictions.items():
        row = {'model': model_name}
        row.update(metric_summary(y_true, y_pred))
        rows.append(row)
    df = pd.DataFrame(rows).set_index('model')
    return df[METRIC_ORDER].sort_values('macro_f1', ascending=False)


def mcnemar_test(y_true: np.ndarray, pred_a: np.ndarray, pred_b: np.ndarray, name_a: str, name_b: str) -> dict[str, float | str]:
    correct_a = pred_a == y_true
    correct_b = pred_b == y_true
    b = int(np.sum(correct_a & ~correct_b))
    c = int(np.sum(~correct_a & correct_b))

    if b + c == 0:
        chi2_stat = 0.0
        p_value = 1.0
    else:
        from scipy.stats import chi2
        chi2_stat = (abs(b - c) - 1) ** 2 / (b + c)
        p_value = 1.0 - chi2.cdf(chi2_stat, df=1)

    return {
        'comparison': f'{name_a} vs {name_b}',
        'a_correct_b_wrong': b,
        'a_wrong_b_correct': c,
        'chi2': chi2_stat,
        'p_value': p_value,
    }


def per_class_metrics_table(predictions: dict[str, np.ndarray], y_true: np.ndarray) -> pd.DataFrame:
    rows = []
    for model_name, y_pred in predictions.items():
        report = classification_report(
            y_true,
            y_pred,
            labels=[0, 1],
            output_dict=True,
            zero_division=0,
        )
        for label in ['0', '1']:
            rows.append(
                {
                    'model': model_name,
                    'class': int(label),
                    'precision': report[label]['precision'],
                    'recall': report[label]['recall'],
                    'f1': report[label]['f1-score'],
                    'support': int(report[label]['support']),
                }
            )
    return pd.DataFrame(rows).set_index(['model', 'class'])


def confusion_matrix_table(y_true: np.ndarray, y_pred: np.ndarray) -> pd.DataFrame:
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    return pd.DataFrame(cm, index=['gold_0', 'gold_1'], columns=['pred_0', 'pred_1'])


official_baseline_df = pd.read_csv(OFFICIAL_BASELINE_PATH)
official_baseline_df = official_baseline_df.rename(columns=lambda col: str(col).strip())
if 'Unnamed: 0' in official_baseline_df.columns:
    official_baseline_df = official_baseline_df.drop(columns=['Unnamed: 0'])

official_baseline_df['reference'] = official_baseline_df['reference'].astype(int)
if official_baseline_df['reference'].tolist() != dev_df['label'].tolist():
    raise ValueError('The reference column in 25_DEV_NLI.csv does not match training_data/NLI/dev.csv.')

official_svm_pred = official_baseline_df['SVM'].astype(int).to_numpy()
y_dev = dev_df['label'].to_numpy(dtype=int)
y_train = train_df['label'].to_numpy(dtype=int)

print('Official baseline methods available:', [col for col in official_baseline_df.columns if col != 'reference'])
pd.DataFrame({'reference': official_baseline_df['reference'].head(5), 'SVM': official_baseline_df['SVM'].head(5)})


Official baseline methods available: ['SVM', 'LSTM', 'BERT']


,reference,SVM
0,0,0
1,1,1
2,1,1
3,0,0
4,1,1


## 4. Internal baseline for this notebook

This is the simple baseline used inside this notebook:

- concatenate `premise` and `hypothesis` with a separator token
- represent the concatenated text with word 1-2 gram TF-IDF
- train a Logistic Regression classifier

This baseline is mainly a sanity check. For the coursework comparison, the more important reference point is the **bundled official Category A baseline** (`SVM`).


In [5]:
baseline_train_text = train_df['premise'].astype(str) + ' [SEP] ' + train_df['hypothesis'].astype(str)
baseline_dev_text = dev_df['premise'].astype(str) + ' [SEP] ' + dev_df['hypothesis'].astype(str)

baseline_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_features=30000,
    sublinear_tf=True,
)
X_baseline_train = baseline_vectorizer.fit_transform(baseline_train_text)
X_baseline_dev = baseline_vectorizer.transform(baseline_dev_text)

internal_baseline_model = LogisticRegression(
    max_iter=2000,
    solver='liblinear',
    random_state=SEED,
)
internal_baseline_model.fit(X_baseline_train, y_train)
internal_baseline_pred = internal_baseline_model.predict(X_baseline_dev)
internal_baseline_metrics = metric_summary(y_dev, internal_baseline_pred)

pd.DataFrame([internal_baseline_metrics], index=['internal_baseline']).T


,internal_baseline
accuracy_score,0.608224
macro_precision,0.608046
macro_recall,0.606064
macro_f1,0.605308
weighted_macro_precision,0.608088
weighted_macro_recall,0.608224
weighted_mmacro_f1,0.606416
matthews_corrcoef,0.214100


## 5. Final Solution A representation

The main Solution A representation stays inside traditional machine learning, but is richer than the internal baseline because it adds pairwise structure.

Feature blocks used by the rich representation:

1. **Premise word TF-IDF**: word 1-2 grams from the premise only
2. **Hypothesis word TF-IDF**: word 1-2 grams from the hypothesis only
3. **Shared-space interactions**: absolute difference and element-wise product after projecting both texts into the same word-TF-IDF space
4. **Pair-level character TF-IDF**: character n-grams from `premise [SEP] hypothesis`
5. **Hand-crafted pair features** computed directly from the provided text:
   - lexical overlap
   - new-token ratio
   - length features
   - negation mismatch
   - number mismatch
   - simple punctuation cues

The richer representation supports several Category A candidates: Logistic Regression, Linear SVM, ablations, bootstrap LR ensembles, and vocabulary-size sensitivity checks.


In [6]:
TOKEN_PATTERN = re.compile(r"[a-z0-9]+(?:'[a-z0-9]+)?")
NUMBER_PATTERN = re.compile(r'\d+(?:\.\d+)?')
NEGATION_TOKENS = {'no', 'not', 'never', 'none', 'nobody', 'nothing', 'neither', 'nor', 'without'}
FULL_FEATURE_BLOCK_ORDER = [
    'premise_word_tfidf',
    'hypothesis_word_tfidf',
    'shared_abs_difference',
    'shared_product',
    'pair_char_tfidf',
    'handcrafted_dense',
]


def normalize_text(text: str) -> str:
    text = str(text)
    text = text.replace('’', "'").replace('‘', "'")
    text = text.replace('“', '"').replace('”', '"')
    return re.sub(r'\s+', ' ', text).strip()


def tokenize(text: str) -> list[str]:
    return TOKEN_PATTERN.findall(normalize_text(text).lower())


HANDCRAFTED_FEATURE_NAMES = [
    'hypothesis_token_recall',
    'premise_token_precision',
    'jaccard',
    'new_token_ratio',
    'premise_length',
    'hypothesis_length',
    'length_ratio',
    'length_difference',
    'premise_has_negation',
    'hypothesis_has_negation',
    'negation_mismatch',
    'shared_number_count',
    'number_mismatch',
    'exact_string_match',
    'hypothesis_token_subset',
    'premise_has_question_mark',
    'hypothesis_has_question_mark',
    'question_mark_delta',
    'premise_has_exclamation_mark',
    'hypothesis_has_exclamation_mark',
    'exclamation_mark_delta',
]


def build_handcrafted_pair_features(df: pd.DataFrame) -> np.ndarray:
    rows = []
    for premise, hypothesis in zip(df['premise'], df['hypothesis']):
        premise_text = normalize_text(premise)
        hypothesis_text = normalize_text(hypothesis)
        premise_tokens = tokenize(premise_text)
        hypothesis_tokens = tokenize(hypothesis_text)
        premise_set = set(premise_tokens)
        hypothesis_set = set(hypothesis_tokens)

        overlap = len(premise_set & hypothesis_set)
        union = len(premise_set | hypothesis_set)
        hypothesis_token_recall = overlap / len(hypothesis_set) if hypothesis_set else 0.0
        premise_token_precision = overlap / len(premise_set) if premise_set else 0.0
        jaccard = overlap / union if union else 0.0
        new_token_ratio = len(hypothesis_set - premise_set) / len(hypothesis_set) if hypothesis_set else 0.0

        premise_length = len(premise_tokens)
        hypothesis_length = len(hypothesis_tokens)
        length_ratio = hypothesis_length / premise_length if premise_length else 0.0
        length_difference = premise_length - hypothesis_length

        premise_has_negation = int(any(tok in NEGATION_TOKENS or tok.endswith("n't") for tok in premise_tokens))
        hypothesis_has_negation = int(any(tok in NEGATION_TOKENS or tok.endswith("n't") for tok in hypothesis_tokens))
        negation_mismatch = int(premise_has_negation != hypothesis_has_negation)

        premise_numbers = NUMBER_PATTERN.findall(premise_text)
        hypothesis_numbers = NUMBER_PATTERN.findall(hypothesis_text)
        shared_number_count = len(set(premise_numbers) & set(hypothesis_numbers))
        number_mismatch = int(bool(premise_numbers or hypothesis_numbers) and set(premise_numbers) != set(hypothesis_numbers))

        exact_string_match = int(premise_text.lower() == hypothesis_text.lower())
        hypothesis_token_subset = int(hypothesis_set.issubset(premise_set)) if hypothesis_set else 0

        premise_has_question_mark = int('?' in premise_text)
        hypothesis_has_question_mark = int('?' in hypothesis_text)
        question_mark_delta = hypothesis_has_question_mark - premise_has_question_mark
        premise_has_exclamation_mark = int('!' in premise_text)
        hypothesis_has_exclamation_mark = int('!' in hypothesis_text)
        exclamation_mark_delta = hypothesis_has_exclamation_mark - premise_has_exclamation_mark

        rows.append([
            hypothesis_token_recall,
            premise_token_precision,
            jaccard,
            new_token_ratio,
            premise_length,
            hypothesis_length,
            length_ratio,
            length_difference,
            premise_has_negation,
            hypothesis_has_negation,
            negation_mismatch,
            shared_number_count,
            number_mismatch,
            exact_string_match,
            hypothesis_token_subset,
            premise_has_question_mark,
            hypothesis_has_question_mark,
            question_mark_delta,
            premise_has_exclamation_mark,
            hypothesis_has_exclamation_mark,
            exclamation_mark_delta,
        ])

    return np.asarray(rows, dtype=np.float32)


def sparse_absolute_difference(left: sp.csr_matrix, right: sp.csr_matrix) -> sp.csr_matrix:
    diff = (left - right).tocsr(copy=True)
    diff.data = np.abs(diff.data)
    return diff


In [7]:
class SolutionAFeatureBuilder:
    def __init__(
        self,
        premise_word_max_features: int = 12000,
        hypothesis_word_max_features: int = 12000,
        shared_word_max_features: int = 8000,
        pair_char_max_features: int = 8000,
    ) -> None:
        self.premise_word_vectorizer = TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=2,
            max_features=premise_word_max_features,
            sublinear_tf=True,
        )
        self.hypothesis_word_vectorizer = TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=2,
            max_features=hypothesis_word_max_features,
            sublinear_tf=True,
        )
        self.shared_word_vectorizer = TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=2,
            max_features=shared_word_max_features,
            sublinear_tf=True,
        )
        self.pair_char_vectorizer = TfidfVectorizer(
            analyzer='char_wb',
            ngram_range=(3, 5),
            min_df=2,
            max_features=pair_char_max_features,
            sublinear_tf=True,
        )
        self.handcrafted_scaler = StandardScaler()
        self.handcrafted_feature_names_ = HANDCRAFTED_FEATURE_NAMES.copy()
        self.feature_block_dimensions_ = {}
        self.config_ = {
            'premise_word_max_features': premise_word_max_features,
            'hypothesis_word_max_features': hypothesis_word_max_features,
            'shared_word_max_features': shared_word_max_features,
            'pair_char_max_features': pair_char_max_features,
        }

    def _normalised_premise_series(self, df: pd.DataFrame) -> pd.Series:
        return df['premise'].fillna('').map(normalize_text)

    def _normalised_hypothesis_series(self, df: pd.DataFrame) -> pd.Series:
        return df['hypothesis'].fillna('').map(normalize_text)

    def fit(self, df: pd.DataFrame) -> 'SolutionAFeatureBuilder':
        premise_text = self._normalised_premise_series(df)
        hypothesis_text = self._normalised_hypothesis_series(df)
        pair_text = premise_text + ' [SEP] ' + hypothesis_text
        handcrafted = build_handcrafted_pair_features(df)

        self.premise_word_vectorizer.fit(premise_text)
        self.hypothesis_word_vectorizer.fit(hypothesis_text)
        self.shared_word_vectorizer.fit(pd.concat([premise_text, hypothesis_text], ignore_index=True))
        self.pair_char_vectorizer.fit(pair_text)
        self.handcrafted_scaler.fit(handcrafted)


        self.feature_block_dimensions_ = {
            'premise_word_tfidf': len(self.premise_word_vectorizer.get_feature_names_out()),
            'hypothesis_word_tfidf': len(self.hypothesis_word_vectorizer.get_feature_names_out()),
            'shared_abs_difference': len(self.shared_word_vectorizer.get_feature_names_out()),
            'shared_product': len(self.shared_word_vectorizer.get_feature_names_out()),
            'pair_char_tfidf': len(self.pair_char_vectorizer.get_feature_names_out()),
            'handcrafted_dense': len(self.handcrafted_feature_names_),
        }
        return self

    def transform(self, df: pd.DataFrame) -> sp.csr_matrix:
        blocks = build_feature_block_matrices(feature_components_from_builder(self), df)
        return stack_selected_blocks(blocks, FULL_FEATURE_BLOCK_ORDER)

    def fit_transform(self, df: pd.DataFrame) -> sp.csr_matrix:
        self.fit(df)
        return self.transform(df)


def feature_components_from_builder(feature_builder: SolutionAFeatureBuilder) -> dict:
    return {
        'premise_word_vectorizer': feature_builder.premise_word_vectorizer,
        'hypothesis_word_vectorizer': feature_builder.hypothesis_word_vectorizer,
        'shared_word_vectorizer': feature_builder.shared_word_vectorizer,
        'pair_char_vectorizer': feature_builder.pair_char_vectorizer,
        'handcrafted_scaler': feature_builder.handcrafted_scaler,
        'handcrafted_feature_names': feature_builder.handcrafted_feature_names_,
        'feature_block_dimensions': feature_builder.feature_block_dimensions_,
        'feature_config': feature_builder.config_,
    }


def build_feature_block_matrices(feature_components: dict, df: pd.DataFrame) -> dict[str, sp.csr_matrix]:
    premise_text = df['premise'].fillna('').map(normalize_text)
    hypothesis_text = df['hypothesis'].fillna('').map(normalize_text)
    pair_text = premise_text + ' [SEP] ' + hypothesis_text

    premise_word = feature_components['premise_word_vectorizer'].transform(premise_text)
    hypothesis_word = feature_components['hypothesis_word_vectorizer'].transform(hypothesis_text)

    shared_premise = feature_components['shared_word_vectorizer'].transform(premise_text)
    shared_hypothesis = feature_components['shared_word_vectorizer'].transform(hypothesis_text)
    shared_abs_difference = sparse_absolute_difference(shared_premise, shared_hypothesis)
    shared_product = shared_premise.multiply(shared_hypothesis)

    pair_char = feature_components['pair_char_vectorizer'].transform(pair_text)
    handcrafted = build_handcrafted_pair_features(df)
    handcrafted_scaled = feature_components['handcrafted_scaler'].transform(handcrafted)

    return {
        'premise_word_tfidf': premise_word,
        'hypothesis_word_tfidf': hypothesis_word,
        'shared_abs_difference': shared_abs_difference,
        'shared_product': shared_product,
        'pair_char_tfidf': pair_char,
        'handcrafted_dense': sp.csr_matrix(handcrafted_scaled),
    }


def stack_selected_blocks(blocks: dict[str, sp.csr_matrix], selected_blocks: list[str]) -> sp.csr_matrix:
    return sp.hstack([blocks[block_name] for block_name in selected_blocks], format='csr')


def transform_with_feature_components(
    feature_components: dict,
    df: pd.DataFrame,
    selected_blocks: list[str] | None = None,
) -> sp.csr_matrix:
    blocks = build_feature_block_matrices(feature_components, df)
    if selected_blocks is None:
        selected_blocks = FULL_FEATURE_BLOCK_ORDER
    return stack_selected_blocks(blocks, selected_blocks)


In [8]:
feature_builder = SolutionAFeatureBuilder()
X_solution_a_train = feature_builder.fit_transform(train_df)
X_solution_a_dev = feature_builder.transform(dev_df)
solution_a_feature_components = feature_components_from_builder(feature_builder)
train_feature_blocks = build_feature_block_matrices(solution_a_feature_components, train_df)
dev_feature_blocks = build_feature_block_matrices(solution_a_feature_components, dev_df)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
param_grid = {
    'C': [0.5, 1.0, 2.0, 4.0],
    'class_weight': [None, 'balanced'],
}

search = GridSearchCV(
    estimator=LogisticRegression(
        max_iter=2000,
        solver='liblinear',
        random_state=SEED,
    ),
    param_grid=param_grid,
    scoring='f1_macro',
    cv=cv,
    n_jobs=1,
    refit=True,
)
search.fit(X_solution_a_train, y_train)

solution_a_model = search.best_estimator_
solution_a_dev_pred = solution_a_model.predict(X_solution_a_dev)
solution_a_metrics = metric_summary(y_dev, solution_a_dev_pred)

print('Best hyperparameters:', search.best_params_)
print('Best CV macro F1:', round(search.best_score_, 4))
print()
pd.DataFrame([solution_a_metrics], index=['rich_feature_logistic_regression']).T


Best hyperparameters: {'C': 0.5, 'class_weight': None}
Best CV macro F1: 0.6875



,rich_feature_logistic_regression
accuracy_score,0.688836
macro_precision,0.688748
macro_recall,0.687718
macro_f1,0.687832
weighted_macro_precision,0.688784
weighted_macro_recall,0.688836
weighted_mmacro_f1,0.688410
matthews_corrcoef,0.376464


## 6. Additional Category A candidates and controlled ablations

This section adds multiple realistic Category A candidates so that the final Solution A can be selected with evidence rather than intuition.

New experiments:

- **Rich feature set + Linear SVM**: checks whether a max-margin classifier is stronger than Logistic Regression on the same representation.
- **Feature ablations**: show which feature blocks actually help, supporting soundness and interpretability.
- **Bootstrap LR ensemble**: adds a creative but still Category A way to improve stability and predictive strength.
- **Feature-size sensitivity**: checks whether the richer representation is robust to vocabulary size choices.

To keep the notebook practical, the rich LR model is tuned once with macro F1. The ablations and size-sensitivity checks then reuse the tuned LR hyperparameters so that differences mainly reflect representation changes rather than repeated hyperparameter searches.


In [9]:
ABLATION_SPECS = {
    'Ablation 1: concat TF-IDF baseline': None,
    'Ablation 2: premise + hypothesis TF-IDF only': [
        'premise_word_tfidf',
        'hypothesis_word_tfidf',
    ],
    'Ablation 3: + shared interaction features': [
        'premise_word_tfidf',
        'hypothesis_word_tfidf',
        'shared_abs_difference',
        'shared_product',
    ],
    'Ablation 4: + char TF-IDF': [
        'premise_word_tfidf',
        'hypothesis_word_tfidf',
        'shared_abs_difference',
        'shared_product',
        'pair_char_tfidf',
    ],
    'Ablation 5: full feature model': FULL_FEATURE_BLOCK_ORDER.copy(),
}

RICH_LR_FIXED_PARAMS = {
    'C': solution_a_model.C,
    'class_weight': solution_a_model.class_weight,
}


def fit_fixed_lr(X_train: sp.csr_matrix, y_train: np.ndarray) -> LogisticRegression:
    model = LogisticRegression(
        max_iter=2000,
        solver='liblinear',
        random_state=SEED,
        C=RICH_LR_FIXED_PARAMS['C'],
        class_weight=RICH_LR_FIXED_PARAMS['class_weight'],
    )
    model.fit(X_train, y_train)
    return model


def register_single_estimator_candidate(
    registry: dict,
    *,
    name: str,
    estimator,
    X_dev: sp.csr_matrix,
    y_dev: np.ndarray,
    feature_components: dict,
    selected_blocks: list[str],
    notes: str,
    deployable: bool,
) -> None:
    predictions = estimator.predict(X_dev).astype(int)
    registry[name] = {
        'predictor_type': 'single_estimator',
        'estimator': estimator,
        'feature_components': feature_components,
        'selected_blocks': selected_blocks,
        'predictions': predictions,
        'metrics': metric_summary(y_dev, predictions),
        'notes': notes,
        'deployable': deployable,
    }


def build_rich_lr_variant(
    train_df: pd.DataFrame,
    dev_df: pd.DataFrame,
    *,
    name: str,
    feature_config: dict,
    deployable: bool = True,
) -> dict:
    builder = SolutionAFeatureBuilder(**feature_config)
    X_train = builder.fit_transform(train_df)
    X_dev = builder.transform(dev_df)
    feature_components = feature_components_from_builder(builder)
    estimator = fit_fixed_lr(X_train, y_train)
    predictions = estimator.predict(X_dev).astype(int)
    return {
        'predictor_type': 'single_estimator',
        'estimator': estimator,
        'feature_components': feature_components,
        'selected_blocks': FULL_FEATURE_BLOCK_ORDER.copy(),
        'predictions': predictions,
        'metrics': metric_summary(y_dev, predictions),
        'notes': f'Rich Logistic Regression variant with feature config {feature_config}',
        'deployable': deployable,
    }


def bootstrap_lr_ensemble(
    X_train: sp.csr_matrix,
    y_train: np.ndarray,
    X_dev: sp.csr_matrix,
    *,
    n_models: int = 5,
    random_state: int = SEED,
) -> tuple[list[LogisticRegression], np.ndarray, np.ndarray]:
    rng = np.random.default_rng(random_state)
    models = []
    probability_sum = None

    for _ in range(n_models):
        sample_idx = rng.integers(0, len(y_train), size=len(y_train))
        model = fit_fixed_lr(X_train[sample_idx], y_train[sample_idx])
        model_proba = model.predict_proba(X_dev)
        probability_sum = model_proba if probability_sum is None else probability_sum + model_proba
        models.append(model)

    average_proba = probability_sum / n_models
    predictions = models[0].classes_[np.argmax(average_proba, axis=1)].astype(int)
    return models, predictions, average_proba


In [10]:
candidate_registry = {
    'Rich-feature Logistic Regression': {
        'predictor_type': 'single_estimator',
        'estimator': solution_a_model,
        'feature_components': solution_a_feature_components,
        'selected_blocks': FULL_FEATURE_BLOCK_ORDER.copy(),
        'predictions': solution_a_dev_pred,
        'metrics': solution_a_metrics,
        'notes': 'Tuned rich-feature Logistic Regression using macro F1 grid search.',
        'deployable': True,
    }
}

ablation_rows = []
for ablation_name, selected_blocks in ABLATION_SPECS.items():
    if selected_blocks is None:
        ablation_pred = internal_baseline_pred
        ablation_metrics = metric_summary(y_dev, ablation_pred)
        ablation_rows.append({'variant': ablation_name, **ablation_metrics})
        continue

    X_variant_train = stack_selected_blocks(train_feature_blocks, selected_blocks)
    X_variant_dev = stack_selected_blocks(dev_feature_blocks, selected_blocks)
    ablation_model = fit_fixed_lr(X_variant_train, y_train)
    ablation_pred = ablation_model.predict(X_variant_dev).astype(int)
    ablation_metrics = metric_summary(y_dev, ablation_pred)
    ablation_rows.append({'variant': ablation_name, **ablation_metrics})

    if ablation_name != 'Ablation 5: full feature model':
        candidate_registry[ablation_name] = {
            'predictor_type': 'single_estimator',
            'estimator': ablation_model,
            'feature_components': solution_a_feature_components,
            'selected_blocks': selected_blocks,
            'predictions': ablation_pred,
            'metrics': ablation_metrics,
            'notes': f'Controlled ablation using blocks {selected_blocks}.',
            'deployable': False,
        }

ablation_results_df = pd.DataFrame(ablation_rows).set_index('variant')[METRIC_ORDER].sort_values('macro_f1', ascending=False)
ablation_results_df


,accuracy_score,macro_precision,macro_recall,macro_f1,weighted_macro_precision,weighted_macro_recall,weighted_mmacro_f1,matthews_corrcoef
variant,,,,,,,,
Ablation 5: full feature model,0.688836,0.688748,0.687718,0.687832,0.688784,0.688836,0.688410,0.376464
Ablation 4: + char TF-IDF,0.672506,0.672650,0.671011,0.670998,0.672604,0.672506,0.671726,0.343657
Ablation 3: + shared interaction features,0.671318,0.671634,0.669676,0.669598,0.671542,0.671318,0.670377,0.341305
Ablation 2: premise + hypothesis TF-IDF only,0.652167,0.652988,0.649976,0.649432,0.652802,0.652167,0.650444,0.302949
Ablation 1: concat TF-IDF baseline,0.608224,0.608046,0.606064,0.605308,0.608088,0.608224,0.606416,0.214100


In [11]:
linear_svm_search = GridSearchCV(
    estimator=LinearSVC(
        max_iter=10000,
        tol=1e-3,
        random_state=SEED,
    ),
    param_grid={
        'C': [1.0],
        'class_weight': [None],
    },
    scoring='f1_macro',
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED),
    n_jobs=1,
    refit=True,
)
linear_svm_search.fit(X_solution_a_train, y_train)

rich_linear_svm = linear_svm_search.best_estimator_
rich_linear_svm_pred = rich_linear_svm.predict(X_solution_a_dev).astype(int)
register_single_estimator_candidate(
    candidate_registry,
    name='Rich-feature Linear SVM',
    estimator=rich_linear_svm,
    X_dev=X_solution_a_dev,
    y_dev=y_dev,
    feature_components=solution_a_feature_components,
    selected_blocks=FULL_FEATURE_BLOCK_ORDER.copy(),
    notes='Linear SVM trained on the same rich feature representation as the tuned LR model.',
    deployable=True,
)

bootstrap_models, bootstrap_pred, bootstrap_avg_proba = bootstrap_lr_ensemble(
    X_solution_a_train,
    y_train,
    X_solution_a_dev,
    n_models=5,
    random_state=SEED,
)
candidate_registry['Bootstrap LR ensemble'] = {
    'predictor_type': 'bootstrap_lr_ensemble',
    'estimators': bootstrap_models,
    'classes': bootstrap_models[0].classes_.astype(int),
    'feature_components': solution_a_feature_components,
    'selected_blocks': FULL_FEATURE_BLOCK_ORDER.copy(),
    'predictions': bootstrap_pred,
    'metrics': metric_summary(y_dev, bootstrap_pred),
    'notes': 'Five-model bootstrap ensemble using the tuned rich-feature Logistic Regression setup.',
    'deployable': True,
}

size_sensitivity_configs = {
    'Rich LR (smaller vocabulary)': {
        'premise_word_max_features': 8000,
        'hypothesis_word_max_features': 8000,
        'shared_word_max_features': 5000,
        'pair_char_max_features': 5000,
    },
    'Rich LR (larger vocabulary)': {
        'premise_word_max_features': 16000,
        'hypothesis_word_max_features': 16000,
        'shared_word_max_features': 10000,
        'pair_char_max_features': 10000,
    },
}

for variant_name, feature_config in size_sensitivity_configs.items():
    candidate_registry[variant_name] = build_rich_lr_variant(
        train_df,
        dev_df,
        name=variant_name,
        feature_config=feature_config,
        deployable=True,
    )

additional_candidate_rows = []
for name in [
    'Rich-feature Logistic Regression',
    'Rich-feature Linear SVM',
    'Bootstrap LR ensemble',
    'Rich LR (smaller vocabulary)',
    'Rich LR (larger vocabulary)',
]:
    additional_candidate_rows.append({'candidate': name, **candidate_registry[name]['metrics']})

print('Linear SVM best hyperparameters:', linear_svm_search.best_params_)
print('Linear SVM best CV macro F1:', round(linear_svm_search.best_score_, 4))
print()
pd.DataFrame(additional_candidate_rows).set_index('candidate')[METRIC_ORDER].sort_values('macro_f1', ascending=False)

/opt/homebrew/lib/python3.11/site-packages/sklearn/svm/_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/opt/homebrew/lib/python3.11/site-packages/sklearn/svm/_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


Linear SVM best hyperparameters: {'C': 1.0, 'class_weight': None}
Linear SVM best CV macro F1: 0.6493



,accuracy_score,macro_precision,macro_recall,macro_f1,weighted_macro_precision,weighted_macro_recall,weighted_mmacro_f1,matthews_corrcoef
candidate,,,,,,,,
Rich-feature Logistic Regression,0.688836,0.688748,0.687718,0.687832,0.688784,0.688836,0.688410,0.376464
Rich LR (smaller vocabulary),0.688391,0.688233,0.687354,0.687473,0.688301,0.688391,0.688026,0.375587
Bootstrap LR ensemble,0.688242,0.688439,0.686851,0.686919,0.688373,0.688242,0.687583,0.375287
Rich LR (larger vocabulary),0.687945,0.687895,0.686777,0.686885,0.687915,0.687945,0.687480,0.374671
Rich-feature Linear SVM,0.650238,0.649988,0.650097,0.650009,0.650431,0.650238,0.650301,0.300085


## 7. Candidate comparison and statistical evaluation

The table below gathers the main Category A candidates, the controlled ablations, and the official same-category baseline in one place.

This directly supports the coursework goals of:

- choosing the strongest final Solution A based on dev evidence
- comparing against the same-category bundled baseline
- showing evaluation effort outside Codabench
- demonstrating that the richer representation is technically justified rather than arbitrary


In [12]:
comparison_predictions = {
    'Official Category A baseline (SVM)': official_svm_pred,
    'Internal baseline (concat TF-IDF + LR)': internal_baseline_pred,
    'Rich-feature Logistic Regression': candidate_registry['Rich-feature Logistic Regression']['predictions'],
    'Rich-feature Linear SVM': candidate_registry['Rich-feature Linear SVM']['predictions'],
    'Bootstrap LR ensemble': candidate_registry['Bootstrap LR ensemble']['predictions'],
    'Ablation 1: concat TF-IDF baseline': internal_baseline_pred,
    'Ablation 2: premise + hypothesis TF-IDF only': candidate_registry['Ablation 2: premise + hypothesis TF-IDF only']['predictions'],
    'Ablation 3: + shared interaction features': candidate_registry['Ablation 3: + shared interaction features']['predictions'],
    'Ablation 4: + char TF-IDF': candidate_registry['Ablation 4: + char TF-IDF']['predictions'],
    'Ablation 5: full feature model': candidate_registry['Rich-feature Logistic Regression']['predictions'],
    'Rich LR (smaller vocabulary)': candidate_registry['Rich LR (smaller vocabulary)']['predictions'],
    'Rich LR (larger vocabulary)': candidate_registry['Rich LR (larger vocabulary)']['predictions'],
}

comparison_df = metrics_table(comparison_predictions, y_dev)
comparison_df


,accuracy_score,macro_precision,macro_recall,macro_f1,weighted_macro_precision,weighted_macro_recall,weighted_mmacro_f1,matthews_corrcoef
model,,,,,,,,
Rich-feature Logistic Regression,0.688836,0.688748,0.687718,0.687832,0.688784,0.688836,0.688410,0.376464
Ablation 5: full feature model,0.688836,0.688748,0.687718,0.687832,0.688784,0.688836,0.688410,0.376464
Rich LR (smaller vocabulary),0.688391,0.688233,0.687354,0.687473,0.688301,0.688391,0.688026,0.375587
Bootstrap LR ensemble,0.688242,0.688439,0.686851,0.686919,0.688373,0.688242,0.687583,0.375287
Rich LR (larger vocabulary),0.687945,0.687895,0.686777,0.686885,0.687915,0.687945,0.687480,0.374671
Ablation 4: + char TF-IDF,0.672506,0.672650,0.671011,0.670998,0.672604,0.672506,0.671726,0.343657
Ablation 3: + shared interaction features,0.671318,0.671634,0.669676,0.669598,0.671542,0.671318,0.670377,0.341305
Rich-feature Linear SVM,0.650238,0.649988,0.650097,0.650009,0.650431,0.650238,0.650301,0.300085
Ablation 2: premise + hypothesis TF-IDF only,0.652167,0.652988,0.649976,0.649432,0.652802,0.652167,0.650444,0.302949


In [13]:
deployable_candidate_names = [
    name
    for name, payload in candidate_registry.items()
    if payload.get('deployable')
]
best_deployable_candidate_name = comparison_df.loc[
    comparison_df.index.intersection(deployable_candidate_names)
].sort_values('macro_f1', ascending=False).index[0]

significance_tests = [
    mcnemar_test(
        y_dev,
        comparison_predictions['Rich-feature Logistic Regression'],
        official_svm_pred,
        'Rich-feature Logistic Regression',
        'Official Category A baseline (SVM)',
    ),
    mcnemar_test(
        y_dev,
        comparison_predictions['Rich-feature Linear SVM'],
        official_svm_pred,
        'Rich-feature Linear SVM',
        'Official Category A baseline (SVM)',
    ),
    mcnemar_test(
        y_dev,
        comparison_predictions['Bootstrap LR ensemble'],
        official_svm_pred,
        'Bootstrap LR ensemble',
        'Official Category A baseline (SVM)',
    ),
    mcnemar_test(
        y_dev,
        comparison_predictions[best_deployable_candidate_name],
        official_svm_pred,
        best_deployable_candidate_name,
        'Official Category A baseline (SVM)',
    ),
    mcnemar_test(
        y_dev,
        comparison_predictions['Bootstrap LR ensemble'],
        comparison_predictions['Rich-feature Logistic Regression'],
        'Bootstrap LR ensemble',
        'Rich-feature Logistic Regression',
    ),
]

significance_df = pd.DataFrame(significance_tests).drop_duplicates(subset=['comparison']).set_index('comparison')
print('Best deployable Solution A candidate on dev:', best_deployable_candidate_name)
print()
significance_df


Best deployable Solution A candidate on dev: Rich-feature Logistic Regression



,a_correct_b_wrong,a_wrong_b_correct,chi2,p_value
comparison,,,,
Rich-feature Logistic Regression vs Official Category A baseline (SVM),1329,638,242.043721,0.000000
Rich-feature Linear SVM vs Official Category A baseline (SVM),1312,881,84.313725,0.000000
Bootstrap LR ensemble vs Official Category A baseline (SVM),1323,636,240.222563,0.000000
Bootstrap LR ensemble vs Rich-feature Logistic Regression,179,183,0.024862,0.874712


## 10. Save the selected final artefacts

The final bundle is chosen from the **deployable** Category A candidates rather than from the explanatory ablations.

This keeps the notebook honest:

- ablations justify the design
- multiple strong A-category candidates are compared fairly
- the saved final artefact corresponds to the strongest deployable A candidate on the dev set


In [14]:
SOLUTION_A_BUNDLE_PATH = ARTEFACT_DIR / 'nli_solution_a_bundle.joblib'
SOLUTION_A_DEV_PRED_PATH = ARTEFACT_DIR / 'nli_solution_a_dev_predictions.csv'
SOLUTION_A_METRICS_PATH = ARTEFACT_DIR / 'nli_solution_a_dev_metrics.csv'

selected_final_candidate = candidate_registry[best_deployable_candidate_name]
solution_a_bundle = {
    'predictor_type': selected_final_candidate['predictor_type'],
    'estimator': selected_final_candidate.get('estimator'),
    'estimators': selected_final_candidate.get('estimators'),
    'classes': selected_final_candidate.get('classes'),
    'feature_components': selected_final_candidate['feature_components'],
    'selected_blocks': selected_final_candidate['selected_blocks'],
    'metadata': {
        'solution_name': 'NLI Solution A',
        'category': 'A',
        'task': 'NLI',
        'selected_final_candidate': best_deployable_candidate_name,
        'rich_lr_best_params': search.best_params_,
        'rich_lr_best_cv_macro_f1': float(search.best_score_),
        'linear_svm_best_params': linear_svm_search.best_params_,
        'linear_svm_best_cv_macro_f1': float(linear_svm_search.best_score_),
        'feature_block_dimensions': selected_final_candidate['feature_components']['feature_block_dimensions'],
        'handcrafted_feature_names': selected_final_candidate['feature_components']['handcrafted_feature_names'],
        'feature_config': selected_final_candidate['feature_components']['feature_config'],
        'selected_blocks': selected_final_candidate['selected_blocks'],
        'notes': selected_final_candidate['notes'],
    },
}

joblib.dump(solution_a_bundle, SOLUTION_A_BUNDLE_PATH)
pd.Series(comparison_predictions[best_deployable_candidate_name], name='label').to_csv(
    SOLUTION_A_DEV_PRED_PATH,
    index=False,
    header=False,
)
comparison_df.to_csv(SOLUTION_A_METRICS_PATH)

print('Selected final candidate:', best_deployable_candidate_name)
print('Saved:', SOLUTION_A_BUNDLE_PATH)
print('Saved:', SOLUTION_A_DEV_PRED_PATH)
print('Saved:', SOLUTION_A_METRICS_PATH)


Selected final candidate: Rich-feature Logistic Regression
Saved: /Users/jiho/Documents/YR3/34812NLU/NLU_CW/solution-a/artifacts_solution_a/nli_solution_a_bundle.joblib
Saved: /Users/jiho/Documents/YR3/34812NLU/NLU_CW/solution-a/artifacts_solution_a/nli_solution_a_dev_predictions.csv
Saved: /Users/jiho/Documents/YR3/34812NLU/NLU_CW/solution-a/artifacts_solution_a/nli_solution_a_dev_metrics.csv
